In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import os
from dotenv import load_dotenv

#load key for hugging face
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

#1. load the doc
loader = PyPDFLoader("attention.pdf")
pdfDoc = loader.load()
print(pdfDoc)

#2. Chunking docs
seperators = ["\n\n","\n"," ",""]
pdfSplitter = RecursiveCharacterTextSplitter(separators=seperators,chunk_size = 1000, chunk_overlap = 100)
pdfChunks = pdfSplitter.split_documents(pdfDoc)
print(len(pdfChunks))
pdfChunksText = [doc.page_content for doc in pdfChunks]

#3. Embeddings
pdfEmbeder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
pdfEmbeddings = pdfEmbeder.embed_documents(pdfChunksText)
print(len(pdfEmbeddings))



In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(documents=pdfChunks,embedding=pdfEmbeder)

print(vectorstore.index)
a1 = vectorstore.similarity_search(query="scaled dot-product attention softmax queries keys value")
# print(type(a1),a1[0].page_content)
a2=vectorstore.similarity_search_with_score(query="scaled dot-product attention softmax queries keys value")
print(len(a2),a2[0][1])
a2=vectorstore.similarity_search_with_relevance_scores(query="scaled dot-product attention softmax queries keys value",score_threshold=0.6)
for ans in a2:
    print(ans[1])

In [16]:
filter_data = {"producer":"pdfTeX-1.40.25"}
a1 = vectorstore.max_marginal_relevance_search(query="scaled dot-product attention softmax queries keys value",fetch_k=5,lambda_mult=0.5,filter=filter_data)

In [17]:
print(len(a1))

4


In [19]:
type(vectorstore.index)

faiss.swigfaiss_avx2.IndexFlatL2

In [22]:
len(pdfChunks)

49

In [21]:
vectorstore.index.ntotal

49

In [23]:
vectorstore.save_local("/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain/langchain_basics")

In [ ]:
vectorstore = vectorstore.load_local(folder_path="/Users/sanvijanvi/AI_Workspace/ML_DL_Projects/genAI_langchain_projects/langchain/langchain_basics")

In [ ]:
import faiss
